# Alita v3.03 SageMaker Serverless Deployment

This notebook deploys the Alita v3.03 classifier to AWS SageMaker Serverless Inference endpoints. It is intended to be run in a SageMaker Notebook instance on the `conda_pytorch_p10` kernel.

**Prerequisites:**
1. Model archive (.mar file) should be available in S3 or locally
2. Appropriate IAM permissions for SageMaker, ECR, and S3

In [ ]:
import boto3
import sagemaker
from sagemaker import get_execution_role
import time
import json
import base64
from datetime import datetime

## Configuration

In [ ]:
# Configuration
model_name = "alitav3"
region = boto3.Session().region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

# ECR repository
ecr_repository = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{model_name}"
image_tag = "latest"
image_uri = f"{ecr_repository}:{image_tag}"

# SageMaker role
role = get_execution_role()

# Endpoint configurations
# NOTE: for batch endpoints, use concurrency of 80, for real-time, use 20
# https://github.com/tnc-ca-geo/animl-api/issues/101
batch_concurrency = 80
# realtime_concurrency = 20  # uncomment to deploy endpoint for realtime inference

print(f"Region: {region}")
print(f"Account ID: {account_id}")
print(f"Image URI: {image_uri}")
print(f"Role: {role}")

## Build and Push Docker Image to ECR

Create a compressed `*.tar.gz` file from the `*.mar` file per requirement of Amazon SageMaker and upload the model to your Amazon S3 bucket.

**Skip this step if the registry is already made and the custom latest pytorch container is already pushed since this step takes a couple of minutes**

In [ ]:
# Create ECR repository if it doesn't exist
ecr_client = boto3.client('ecr')

try:
    ecr_client.create_repository(repositoryName=model_name)
    print(f"Created ECR repository: {model_name}")
except ecr_client.exceptions.RepositoryAlreadyExistsException:
    print(f"ECR repository {model_name} already exists")

In [ ]:
# Download the model .mar file from S3, create a .tar.gz file, and upload to SageMaker S3 bucket
model_uri = f's3://animl-model-zoo/{model_name}/exported-model/{model_name}.mar'
sagemaker_session = sagemaker.Session(boto_session=boto3.Session())
bucket_name = sagemaker_session.default_bucket()
prefix = 'torchserve'
prod_model_uri = f"s3://{bucket_name}/{prefix}/models/"

!aws s3 cp {model_uri} ./
!tar cvfz {model_name}.tar.gz {model_name}.mar
!aws s3 cp {model_name}.tar.gz {prod_model_uri}

In [ ]:
# Build and push Docker image
import subprocess
import os

# Change to model directory (adjust path as needed)
model_dir = "/home/ec2-user/SageMaker/animl-ml/models/alitav3"
os.chdir(model_dir)

# Get ECR login token
login_cmd = f"aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {ecr_repository}"
subprocess.run(login_cmd, shell=True, check=True)

# Build image
build_cmd = f"docker build -t {model_name}:{image_tag} ."
subprocess.run(build_cmd, shell=True, check=True)

# Tag image
tag_cmd = f"docker tag {model_name}:{image_tag} {image_uri}"
subprocess.run(tag_cmd, shell=True, check=True)

# Push image
push_cmd = f"docker push {image_uri}"
subprocess.run(push_cmd, shell=True, check=True)

print(f"Successfully pushed image to {image_uri}")

## Create SageMaker Model

In [ ]:
sagemaker_client = boto3.client('sagemaker')

# Check if model already exists
model_data = f"{prod_model_uri}{model_name}.tar.gz"
model_already_created = False
for model_def in sagemaker_client.list_models()['Models']:
    if model_name == model_def['ModelName']:
        create_model_response = model_def
        model_already_created = True
        print(f"Model already exists: {model_name}")

# Create model if it doesn't exist
if not model_already_created:
    container = {"Image": image_uri, "ModelDataUrl": model_data}

    if not model_already_created:
        create_model_response = sagemaker_client.create_model(
            ModelName=model_name, ExecutionRoleArn=role, PrimaryContainer=container
        )

    print(create_model_response["ModelArn"])
    print(f"Created model: {model_name}")

print(f"Model ARN: {create_model_response['ModelArn']}")

## Create Serverless Endpoint Configurations

In [ ]:
# Create realtime and batch endpoint configuration
# for batch endpoints, use concurrency of 80, for real-time endpoints, use 20
# https://github.com/tnc-ca-geo/animl-api/issues/101

if batch_concurrency:
    batch_endpoint_config_name = f"{model_name}-config-concurrency-{batch_concurrency}"
    batch_endpoint_config_response = sagemaker_client.create_endpoint_config(
        EndpointConfigName=batch_endpoint_config_name,
        ProductionVariants=[
            {
                "ModelName": model_name,
                "VariantName": "AllTraffic",
                "ServerlessConfig": {
                    "MemorySizeInMB": 4096,  # 4GB memory
                    "MaxConcurrency": batch_concurrency    # Maximum concurrent invocations
                }
            }
        ]
    )
    print(f"Endpoint Config ARN: {batch_endpoint_config_response['EndpointConfigArn']}")


In [ ]:
# If necessary, create realtime endpoint config
if realtime_concurrency:
    realtime_endpoint_config_name = f"{model_name}-config-concurrency-{realtime_concurrency}"
    realtime_endpoint_config_response = sagemaker_client.create_endpoint_config(
        EndpointConfigName=realtime_endpoint_config_name,
        ProductionVariants=[
            {
                "ModelName": model_name,
                "VariantName": "AllTraffic",
                "ServerlessConfig": {
                    "MemorySizeInMB": 4096,  # 4GB memory
                    "MaxConcurrency": realtime_concurrency    # Maximum concurrent invocations
                }
            }
        ]
    )
    print(f"Endpoint Config ARN: {realtime_endpoint_config_response['EndpointConfigArn']}")
else:
    print("`realtime_concurrency` not defined, so skipping. If you need to support real-time inference, uncomment and define `realtime_concurrency` in the Configuration of this Notebook")

## Create Serverless Endpoints

In [ ]:
# Create batch endpoint
batch_endpoint_name = f"{model_name}-concurrency-{batch_concurrency}"
try:
    batch_endpoint_response = sagemaker_client.create_endpoint(
        EndpointName=batch_endpoint_name,
        EndpointConfigName=batch_endpoint_config_name
    )
    print(f"Creating batch endpoint: {batch_endpoint_name}")
except Exception as e:
    if "already exists" in str(e):
        print(f"Batch endpoint {batch_endpoint_name} already exists")
    else:
        raise e

In [ ]:
# Create real-time endpoint (if needed)
if realtime_concurrency:
    realtime_endpoint_name = f"{model_name}-realtime-concurrency-{realtime_concurrency}"
    try:
        realtime_endpoint_response = sagemaker_client.create_endpoint(
            EndpointName=realtime_endpoint_name,
            EndpointConfigName=realtime_endpoint_config_name
        )
        print(f"Creating real-time endpoint: {realtime_endpoint_name}")
    except Exception as e:
        if "already exists" in str(e):
            print(f"Real-time endpoint {realtime_endpoint_name} already exists")
        else:
            raise e
else:
    print("`realtime_concurrency` not defined, so skipping. If you need to support real-time inference, uncomment and define `realtime_concurrency` in the Configuration of this Notebook")

## Wait for Endpoints to be Ready

In [ ]:
def wait_for_endpoint(endpoint_name, timeout_minutes=20):
    """Wait for endpoint to be in service"""
    print(f"Waiting for endpoint {endpoint_name} to be ready...")
    
    start_time = time.time()
    timeout_seconds = timeout_minutes * 60
    
    while True:
        response = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
        status = response['EndpointStatus']
        
        if status == 'InService':
            print(f"Endpoint {endpoint_name} is ready!")
            return True
        elif status == 'Failed':
            print(f"Endpoint {endpoint_name} failed to deploy")
            print(f"Failure reason: {response.get('FailureReason', 'Unknown')}")
            return False
        
        elapsed = time.time() - start_time
        if elapsed > timeout_seconds:
            print(f"Timeout waiting for endpoint {endpoint_name}")
            return False
        
        print(f"  Status: {status} (elapsed: {elapsed/60:.1f}min)")
        time.sleep(30)

# Wait for both endpoints
batch_ready = False
realtime_ready = False
batch_ready = wait_for_endpoint(batch_endpoint_name)
if realtime_concurrency:
    realtime_ready = wait_for_endpoint(realtime_endpoint_name)

## Test Endpoints

In [ ]:
from io import BytesIO
import boto3
from PIL import Image
import json

# Load a test image from the test-data directory
print("Loading test image...")
endpoint_name = batch_endpoint_name
test_image = Image.open("tests/test-data/stoat-test.jpg")
# test_image = Image.open("tests/test-data/rat-test.jpg")
# test_image = Image.open("tests/test-data/kea-test.jpg")
display(test_image)

# Convert image to base64
buffered = BytesIO()
test_image.save(buffered, format="JPEG")
img_str = base64.b64encode(buffered.getvalue()).decode()

# Prepare payload
payload = {
    "image": img_str,
    "bbox": [0.48077553510665894, 0.3209689259529114, 0.690616250038147, 0.5912476778030396] # bbox for stoat-test.jpg
    # "bbox": [0.4617251753807068, 0.01082997303456068, 0.6275747418403625, 0.3216787576675415] # bbox for rat-test.jpg
    # "bbox": [0.5261203050613403, 0.00028116704197600484, 0.9672221541404724, 0.40589991211891174] # bbox for kea-test.jpg

}

# Invoke endpoint
print("Requesting prediction...")
client = boto3.client('runtime.sagemaker')
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='multipart/form-data',
    Body=json.dumps(payload)
)

# Parse results
result = json.loads(response['Body'].read().decode())
print("\nPrediction Results:")
print(json.dumps(result, indent=2))

## Summary and Next Steps

In [ ]:
print("\n=== Deployment Summary ===")
print(f"Model: {model_name}")
print(f"Image URI: {image_uri}")

if batch_ready:
    print(f"Batch Endpoint: {batch_endpoint_name} (concurrency: {batch_concurrency})")
else:
    print(f"Batch Endpoint: {batch_endpoint_name} - Failed or not ready")

if realtime_ready:
    print(f"Real-time Endpoint: {realtime_endpoint_name} (concurrency: {realtime_concurrency})")
else:
    print(f"Real-time Endpoint Failed or was not deployed")

print("\n=== Next Steps ===")
print("1. Update SSM parameters in AWS Systems Manager:")
print(f"   /ml/{model_name}-batch-endpoint-dev: {batch_endpoint_name}")
print(f"   /ml/{model_name}-batch-endpoint-prod: {batch_endpoint_name}")
if realtime_ready:
    print(f"   /ml/{model_name}-realtime-endpoint-dev: {realtime_endpoint_name}")
    print(f"   /ml/{model_name}-realtime-endpoint-prod: {realtime_endpoint_name}")
print("\n2. Update animl-api to integrate the new model")
print("\n3. Add MLModel record to MongoDB with species categories")